In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# split fin

df = pd.read_csv('../data/processed/data_clean.csv')
categorical_cols = df.select_dtypes(include=['str']).columns

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary'), categorical_cols)
    ],
    remainder='passthrough'
)

scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

model = XGBClassifier(
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss'
)

pipe = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', model)
])

param_grid = {
    'model__n_estimators': [200, 400, 600],
    'model__max_depth': [3, 4, 5],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.8, 1],
    'model__colsample_bytree': [0.8, 1]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

In [ ]:
print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred, normalize='true',))

## Probar threeshold

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

proba = best_model.predict_proba(X_test)[:, 1]

results = []
thresholds = np.arange(0.25, 0.85, 0.05)
for threshold in thresholds:
    y_pred_threshold = (proba >= threshold).astype(int)
    precision = precision_score(y_test, y_pred_threshold, pos_label=1)
    recall = recall_score(y_test, y_pred_threshold, pos_label=1)
    f1 = f1_score(y_test, y_pred_threshold, pos_label=1)
    results.append((threshold, precision, recall, f1))
    
df_threshold = pd.DataFrame(results, columns=['Threshold', 'Precision', 'Recall', 'F1'])
df_probs = pd.DataFrame({
    'Real': y_test,
    'Probabilidad': best_model.predict_proba(X_test)[:, 1]
})

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df_probs[df_probs['Real'] == 0], x='Probabilidad', 
             color='skyblue', label='Clase 0 (Real)', kde=True, bins=30, alpha=0.6)

sns.histplot(data=df_probs[df_probs['Real'] == 1], x='Probabilidad', 
             color='orange', label='Clase 1 (Real)', kde=True, bins=30, alpha=0.6)

plt.axvline(x=0.5, color='red', linestyle='--',)

plt.title('Distribución de Probabilidades por Clase Real')
plt.xlabel('Probabilidad de ser Clase 1')
plt.ylabel('Cantidad de casos')
plt.xticks(np.arange(0, 1.05, 0.05), rotation=45)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
df_threshold.plot(x='Threshold', y=['Precision', 'Recall', 'F1'], marker='o')
plt.axvline(x=0.5, color='red', linestyle='--',)
plt.xticks(thresholds, rotation=45)
plt.yticks(np.arange(0, 1.1, 0.1))
plt.show()

In [ ]:
df_threshold.sort_values('F1', ascending=False)

# Optuna

In [ ]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split

THRESHOLD = 0.6

for col in categorical_cols:
    X_train[col] = X_train[col].astype('category').cat.codes
    X_test[col] = X_test[col].astype('category').cat.codes

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "random_state": 42,
        "n_jobs": -1,
        "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
        "eval_metric": "logloss"
    }
    
    # Al ser ahora números enteros, ya no necesitamos enable_categorical=True
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    
    proba = model.predict_proba(X_test)[:, 1]
    y_pred = (proba >= THRESHOLD).astype(int) 
    
    return recall_score(y_test, y_pred, pos_label=1)

# Ejecutar Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("-" * 30)
print("Best Params:", study.best_params)
print("Best Recall:", study.best_value)